[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 04](README.md)

# MPI colectivas, datatypes y topologías

**Tema:** 04 · **Sesiones:** 19, 20 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Qué patrón colectivo expresa la comunicación y cómo cambia su costo con procesos y datos?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** Una colectiva expresa una intención global que la biblioteca puede implementar mejor que una secuencia manual, siempre que counts, layouts y orden sean compatibles.

**Prerrequisitos.**

- Procesos, memoria privada y paso de argumentos.
- Modelo de costo latencia–ancho de banda y referencia serial.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Seleccionar broadcast, scatter/gather, reduce/allreduce.
- Modelar costo con latencia y ancho de banda.
- Calcular vecinos de una topología cartesiana.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Las colectivas deben invocarse en orden compatible por todos los procesos del comunicador.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

Un datatype derivado describe layout; no convierte automáticamente tipos ni corrige extensiones erróneas.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

Las topologías asocian estructura lógica y pueden facilitar mapeo, sin garantizar colocación física.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- rank — identidad de un proceso dentro de un comunicador
- tag — etiqueta que participa en el emparejamiento de mensajes
- colectiva — operación coordinada por todos los procesos del comunicador


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Mpi Comunicacion

![Procesos MPI con mensajes y colectiva](../../images/mpi-comunicacion.svg)

**Cómo leerlo.** Las flechas exteriores representan punto a punto; las interiores, coordinación colectiva. Todos los ranks deben respetar comunicador, orden y contrato de datos.

### Distribucion Trabajo

![Iteraciones distribuidas y reducción final](../../images/distribucion-trabajo.svg)

**Cómo leerlo.** Verifica dos propiedades: cada iteración pertenece a un trabajador y la combinación de parciales reproduce la referencia serial.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "04"
NOTEBOOK = "04_mpi/02_colectivas_topologias.ipynb"
assert (ROOT / "curso" / "notebooks" / "04_mpi" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Costo de colectivas

**Situación.** Se compara un modelo lineal con uno arbóreo para broadcast.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
import math
latency_us, bandwidth_gbs, bytes_ = 2.0, 12.0, 8_000_000
transfer_us = bytes_ / (bandwidth_gbs * 1e9) * 1e6
assert transfer_us > latency_us
for p in (2, 4, 8, 16, 32):
    linear = (p-1) * (latency_us + transfer_us)
    tree = math.ceil(math.log2(p)) * (latency_us + transfer_us)
    assert tree <= linear
    print(f"p={p:2} lineal={linear:9.1f}us árbol={tree:9.1f}us")


### Explicación del resultado

El modelo orienta la hipótesis; la biblioteca puede segmentar, usar árboles distintos y adaptar el algoritmo al tamaño.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Vecinos cartesianos

**Situación.** Se enumeran coordenadas y vecinos sin periodicidad en una malla 3×4.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
rows, cols = 3, 4
def rank(r, c): return r*cols+c if 0 <= r < rows and 0 <= c < cols else None
assert rank(0, 0) == 0 and rank(2, 3) == 11 and rank(-1, 0) is None
for r in range(rows):
    for c in range(cols):
        neighbors = {"N": rank(r-1,c), "S": rank(r+1,c), "W": rank(r,c-1), "E": rank(r,c+1)}
        print(rank(r,c), (r,c), neighbors)


### Lectura razonada

En MPI, `MPI_Cart_shift` obtiene vecinos de acuerdo con dimensiones, periodicidad y posible reordenamiento.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Cuándo `Allreduce` comunica más datos de los necesarios y qué alternativa expresa mejor el patrón?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Reemplazar una secuencia manual por la colectiva equivalente.
2. Verificar counts y desplazamientos para tamaños irregulares.
3. Medir colectiva por tamaño de mensaje y número de procesos.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Invocar colectivas en órdenes distintos.
- Suponer que reduce entrega resultado a todos.
- Crear datatype sin revisar extent.


## Criterios de aceptación

- Orden colectivo compatible.
- Counts, tipos y buffers válidos en cada rango.
- Modelo de costo contrastado con datos.


## Síntesis

- La pregunta que debes poder responder es: **¿Qué patrón colectivo expresa la comunicación y cómo cambia su costo con procesos y datos?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Ejemplos MPI](../../../mpi/)
- [Planeación MPI](../../../docs/PLANEACION_CURSO.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 04](README.md)
